# Group Q3 2025's sessions by channel

*Data Analysis in Python · Topic 4.2*

The social team says Organic Social is missing from the new channel report, and the video team says the same about YouTube. Answer for July to September 2025.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell and the cell that builds the `sessions` table: the clean sessions for July to September 2025, with lesson 3.'s steps.
2. In the first cell marked ✏️, paste your `channel_group()` from Foundations topic 12.3., with the constants and helpers it needs (`SOCIAL_SITES`, `clean()`).
3. In the second cell marked ✏️, write `row_channel_group(row)`, which returns the channel group of one row of the `sessions` table, from the export's own `session_source` and `session_medium`. Add a `channel_group` column to the `sessions` table with `apply()`.
4. In the last cell marked ✏️, create:
   - `unassigned_rows`: the number of rows of the `sessions` table whose channel group is `Unassigned`
   - `not_in_crm`: the set of channel groups in the `sessions` table that aren't in `CUSTOMER_CHANNELS`
   - `organic_social_sessions` and `paid_video_sessions`: the total sessions of the two groups
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `channel_group()` follows Foundations topic 12.3.'s spec. The `sessions` table has a `channel_group` column with nine groups. `unassigned_rows` is `0` and `not_in_crm` is an empty set. `organic_social_sessions` is `16652` and `paid_video_sessions` is `35517`.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2025-07", "2025-08", "2025-09"]
START, END = "2025-07-01", "2025-09-30"
# The CRM's first-touch channels, from customers.csv (Foundations topic 12.3)
CUSTOMER_CHANNELS = {"Organic Search", "Paid Search", "Direct", "Email", "Paid Social", "Paid Video", "Organic Social", "Referral", "Display"}

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

## Your channel grouping

In [ ]:
SOCIAL_SITES = {"facebook.com", "m.facebook.com", "l.facebook.com", "lm.facebook.com", "instagram.com", "l.instagram.com"}


def clean(value):
    """Return the value lowercased and stripped, or "" for None."""
    if value is None:
        return ""
    return value.strip().lower()


def channel_group(source, medium):
    """Return the channel group of a session's source and medium.

    The values are compared lowercased and stripped, and the spec's first
    matching rule wins. "Unassigned" if no rule matches, such as when the
    medium is None.
    """
    source = clean(source)
    medium = clean(medium)
    if source == "(direct)" and medium == "(none)":
        return "Direct"
    if medium == "email":
        return "Email"
    if medium == "cpc":
        return "Paid Search"
    if medium == "display":
        return "Display"
    if medium == "paid_social":
        return "Paid Social"
    if medium == "video":
        return "Paid Video"
    if medium == "organic":
        return "Organic Search"
    if medium == "referral" and source in SOCIAL_SITES:
        return "Organic Social"
    if medium == "referral":
        return "Referral"
    return "Unassigned"

In [ ]:
def row_channel_group(row):
    """Return the channel group of one row of the sessions table."""
    return channel_group(row["session_source"], row["session_medium"])


sessions["channel_group"] = sessions.apply(row_channel_group, axis=1)
sessions["channel_group"].value_counts()

## Check the groups

In [ ]:
unassigned_rows = (sessions["channel_group"] == "Unassigned").sum()
not_in_crm = set(sessions["channel_group"].unique()) - CUSTOMER_CHANNELS
organic_social_sessions = sessions.loc[sessions["channel_group"] == "Organic Social", "sessions"].sum()
paid_video_sessions = sessions.loc[sessions["channel_group"] == "Paid Video", "sessions"].sum()
unassigned_rows, not_in_crm, organic_social_sessions, paid_video_sessions

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    # Foundations topic 12.3's spec, as (source, medium, channel group)
    spec = [("(direct)", "(none)", "Direct"), ("google", "cpc", "Paid Search"), ("google", "display", "Display"), ("youtube", "video", "Paid Video"),
            ("google", "organic", "Organic Search"), ("bing", "organic", "Organic Search"), ("newsletter", "email", "Email"),
            ("Facebook", "paid_social", "Paid Social"), ("fb", "paid_social", "Paid Social"), ("ig", "paid_social", "Paid Social"),
            ("facebook.com", "referral", "Organic Social"), ("l.instagram.com", "referral", "Organic Social"), ("m.facebook.com", "referral", "Organic Social"),
            ("stylebloggers.example", "referral", "Referral"), ("qa", "test", "Unassigned"), ("google", None, "Unassigned"), (None, "cpc", "Paid Search")]
    function = found.get("channel_group")
    if not callable(function):
        results.append((False, "channel_group isn't a function yet. Paste it from Foundations topic 12.3, with SOCIAL_SITES and clean()."))
    else:
        wrong = []
        for source, medium, expected in spec:
            try:
                got = function(source, medium)
            except Exception as error:
                got = f"{type(error).__name__}"
            if got != expected:
                wrong.append(f"{source} / {medium} gave {got!r}, not {expected!r}")
        if not wrong:
            results.append((True, "channel_group() follows Foundations topic 12.3's spec, missing values included."))
        elif any("display" in item or "video" in item for item in wrong):
            results.append((False, f"channel_group() has no rule for Display or Paid Video: {wrong[0]}. That's topic 5.3's first version. Paste the one from topic 12.3."))
        else:
            results.append((False, f"channel_group() doesn't follow topic 12.3's spec: {'; '.join(wrong[:3])}."))

    sessions = found.get("sessions")
    if not isinstance(sessions, pd.DataFrame) or len(sessions) != 5373 or sessions["sessions"].sum() != 333079:
        results.append((False, "sessions should be the clean table for July to September 2025, 5,373 rows. Run the cell that builds it, and don't filter it."))
    elif "channel_group" not in sessions.columns:
        results.append((False, "sessions has no channel_group column yet. Add it: sessions[\"channel_group\"] = sessions.apply(row_channel_group, axis=1)."))
    elif sessions.loc[sessions["channel_group"] == "Organic Social", "sessions"].sum() == 0 and sessions.loc[sessions["channel_group"] == "Referral", "sessions"].sum() == 26247:
        results.append((False, "sessions has no Organic Social, and Referral has 26,247 sessions: row_channel_group() gives the function the standardized source column. SOCIAL_SITES lists the export's spellings, such as facebook.com: use row[\"session_source\"]."))
    elif sessions["channel_group"].nunique() != 9:
        results.append((False, f"sessions has {sessions['channel_group'].nunique()} channel groups, but it should have nine: {sorted(sessions['channel_group'].unique())}."))
    else:
        results.append((True, "sessions has a channel_group column with the nine groups."))

    unassigned = found.get("unassigned_rows")
    not_in_crm = found.get("not_in_crm")
    if isinstance(unassigned, (pd.DataFrame, pd.Series)):
        results.append((False, f"unassigned_rows is a {type(unassigned).__name__}, not a number. Count the rows: (sessions[\"channel_group\"] == \"Unassigned\").sum()."))
    elif unassigned == 774:
        results.append((False, "unassigned_rows is 774: every Display and Paid Video row. Your channel_group() is topic 5.3's version."))
    elif unassigned != 0:
        results.append((False, f"unassigned_rows should be (sessions[\"channel_group\"] == \"Unassigned\").sum(), which is 0. It is {unassigned!r}."))
    elif not isinstance(not_in_crm, set):
        results.append((False, f"not_in_crm should be a set: set(sessions[\"channel_group\"].unique()) - CUSTOMER_CHANNELS. It is {not_in_crm!r}."))
    elif not_in_crm:
        results.append((False, f"not_in_crm is {not_in_crm}: these names aren't in the CRM. Check them against CUSTOMER_CHANNELS."))
    else:
        results.append((True, "No row is Unassigned, and every channel group is a name the CRM uses."))

    social = found.get("organic_social_sessions")
    video = found.get("paid_video_sessions")
    if tables := [name for name, value in [("organic_social_sessions", social), ("paid_video_sessions", video)] if isinstance(value, (pd.DataFrame, pd.Series))]:
        results.append((False, f"{' and '.join(tables)} should be numbers, not tables of rows. Total one column: sessions.loc[sessions[\"channel_group\"] == \"Organic Social\", \"sessions\"].sum()."))
    elif social == 16652 and video == 35517:
        results.append((True, "Organic Social had 16,652 sessions and Paid Video 35,517, from July to September 2025."))
    elif social == 0:
        results.append((False, "organic_social_sessions is 0: the function got the standardized source column. Use session_source."))
    elif video == 0:
        results.append((False, "paid_video_sessions is 0: YouTube's video sessions are Unassigned. Use topic 12.3's channel_group()."))
    else:
        results.append((False, f"organic_social_sessions and paid_video_sessions should be 16652 and 35517, but they are {social!r} and {video!r}."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
SOCIAL_SITES = {"facebook.com", "m.facebook.com", "l.facebook.com", "lm.facebook.com", "instagram.com", "l.instagram.com"}


def clean(value):
    """Return the value lowercased and stripped, or "" for None."""
    if value is None:
        return ""
    return value.strip().lower()


def channel_group(source, medium):
    """Return the channel group of a session's source and medium.

    The values are compared lowercased and stripped, and the spec's first
    matching rule wins. "Unassigned" if no rule matches, such as when the
    medium is None.
    """
    source = clean(source)
    medium = clean(medium)
    if source == "(direct)" and medium == "(none)":
        return "Direct"
    if medium == "email":
        return "Email"
    if medium == "cpc":
        return "Paid Search"
    if medium == "display":
        return "Display"
    if medium == "paid_social":
        return "Paid Social"
    if medium == "video":
        return "Paid Video"
    if medium == "organic":
        return "Organic Search"
    if medium == "referral" and source in SOCIAL_SITES:
        return "Organic Social"
    if medium == "referral":
        return "Referral"
    return "Unassigned"
```

```python
def row_channel_group(row):
    """Return the channel group of one row of the sessions table."""
    return channel_group(row["session_source"], row["session_medium"])


sessions["channel_group"] = sessions.apply(row_channel_group, axis=1)
sessions["channel_group"].value_counts()
```

```python
unassigned_rows = (sessions["channel_group"] == "Unassigned").sum()
not_in_crm = set(sessions["channel_group"].unique()) - CUSTOMER_CHANNELS
organic_social_sessions = sessions.loc[sessions["channel_group"] == "Organic Social", "sessions"].sum()
paid_video_sessions = sessions.loc[sessions["channel_group"] == "Paid Video", "sessions"].sum()
unassigned_rows, not_in_crm, organic_social_sessions, paid_video_sessions
```

</details>